# Bonus: ASR + Language Consistency Analysis

The assignment explicitly calls out bonus value for:
- language ID
- ASR-based quality checks
- visual analysis. fileciteturn2file0

This notebook focuses on the `review` bucket and extracts extra evidence from:
- Whisper language detection
- ASR average log-probability
- final-score distributions for matched vs mismatched language predictions


In [ ]:
%%capture
!pip install -q polars pandas plotly faster-whisper


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/indic_audio_filter"
RUN_DIR = f"{DRIVE_DIR}/outputs/run_v1"


In [ ]:
import polars as pl, os
review_path = f"{RUN_DIR}/review_manifest.jsonl"
df = pl.read_ndjson(review_path) if os.path.exists(review_path) else pl.DataFrame([])
print("Review rows:", len(df))
df.head(5)


In [ ]:
if len(df) > 0:
    pdf = df.to_pandas()
    if 'asr_avg_logprob' in pdf.columns and 'final_score' in pdf.columns:
        import plotly.express as px
        fig = px.scatter(pdf, x='final_score', y='asr_avg_logprob', color='language' if 'language' in pdf.columns else None, hover_data=['sample_id'] if 'sample_id' in pdf.columns else None, title='Review bucket: ASR avg log-prob vs final score')
        fig.show()
    else:
        print("ASR fields not found.")
else:
    print("No review rows found.")


In [ ]:
if len(df) > 0 and 'language_match' in df.columns:
    pdf = df.to_pandas()
    print(pdf['language_match'].value_counts(dropna=False))
    if 'final_score' in pdf.columns:
        import plotly.express as px
        fig = px.box(pdf, x='language_match', y='final_score', title='Final score by language-match flag')
        fig.show()


## How to use this in the final write-up

Explain that the **review bucket** is deliberate:
- hard rejects handle catastrophic failures
- weighted scoring handles normal cases
- ASR/LID helps prioritize ambiguous multilingual samples for re-checking

That shows strong engineering judgment.
